# Zajęcia 6 — testowanie systemów ML

Notebook buduje zestaw testów dla projektu zawierającego cztery ukryte defekty: wyciek etykiety, zbyt słaby kontrakt danych, model poniżej progu jakości i niezgodny kontrakt odpowiedzi.

Testy uruchamiamy jedną komendą, w osobnym procesie — dokładnie tak, jak zrobi to potok CI.

Komórki opisane jako **zadanie** wypełniasz samodzielnie. Komórka pod nimi zawiera rozwiązanie ukryte magią `%%skip True`.

## 1. Przygotowanie notebooka

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
import subprocess
import sys
import textwrap
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "skip_kernel_extension.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

PROJECT_DIR = REPO_ROOT / "artifacts" / "zajecia_06" / "projekt"
for sub in ("src", "tests"):
    (PROJECT_DIR / sub).mkdir(parents=True, exist_ok=True)


def write_file(relative_path: str, content: str) -> Path:
    path = PROJECT_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")
    return path


def run_pytest(*extra: str) -> int:
    started = time.perf_counter()
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *extra],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
    )
    print(result.stdout[-4000:])
    print(f"Kod wyjścia: {result.returncode}, czas: {time.perf_counter() - started:.1f} s")
    return result.returncode


write_file(
    "pytest.ini",
    """
    [pytest]
    pythonpath = src
    testpaths = tests
    addopts = -q
    markers =
        slow: test wymagający dłuższego czasu wykonania
    """,
)
print(f"Projekt: {PROJECT_DIR.relative_to(REPO_ROOT)}")

%load_ext skip_kernel_extension

## 2. Projekt z defektami

Zapisujemy cztery moduły. Każdy zawiera jeden defekt oznaczony komentarzem `DEFEKT`. Nie poprawiaj ich teraz — najpierw napiszemy testy, które je wykryją.

Ta komórka jest gotowa — uruchom ją bez zmian.

In [ ]:
write_file(
    "src/features.py",
    """
    # Transformacja cech.
    from __future__ import annotations

    import pandas as pd

    FEATURE_COLUMNS = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)",
    ]
    TARGET_COLUMN = "target"


    def build_features(frame: pd.DataFrame) -> pd.DataFrame:
        # DEFEKT 1: do zbioru cech trafia kolumna z etykietą.
        return frame[FEATURE_COLUMNS + [TARGET_COLUMN]]
    """,
)

write_file(
    "src/contract.py",
    """
    # Kontrakt wejścia i wyjścia.
    import pandera.pandas as pa

    OUTPUT_KEYS = ("prediction", "model_version")

    # DEFEKT 2: schemat nie sprawdza typów ani kolumn nadmiarowych.
    INPUT_SCHEMA = pa.DataFrameSchema(
        columns={
            "sepal length (cm)": pa.Column(),
            "sepal width (cm)": pa.Column(),
        },
        strict=False,
    )
    """,
)

write_file(
    "src/model.py",
    """
    # Definicja modelu.
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    SEED = 42


    def build_model() -> Pipeline:
        # DEFEKT 3: model jest nadmiernie regularyzowany i nie uczy się niczego.
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(C=1e-6, max_iter=500, random_state=SEED)),
            ]
        )
    """,
)

write_file(
    "src/service.py",
    """
    # Warstwa predykcji udostępniana na zewnątrz.
    from __future__ import annotations

    import pandas as pd

    from features import FEATURE_COLUMNS

    MODEL_VERSION = "0.1.0"


    def predict(model, records: list[dict]) -> list[dict]:
        frame = pd.DataFrame.from_records(records)[FEATURE_COLUMNS].astype(float)
        predictions = model.predict(frame)
        # DEFEKT 4: odpowiedź łamie kontrakt — brak wersji modelu i zły typ predykcji.
        return [{"prediction": float(value)} for value in predictions]
    """,
)

print("Zapisano cztery moduły z defektami.")

## 3. Zestaw testów

Testy piszemy przeciwko **wymaganiom**, a nie przeciwko obecnemu zachowaniu kodu. Dlatego powstają zanim naprawimy defekty — i dlatego mają być czerwone.

**Zadanie:** uzupełnij plik testów o pozostałe przypadki:

- cechy nie zawierają kolumny z etykietą i są liczbowe;
- kontrakt odrzuca błędny typ kolumny oraz kolumnę nadmiarową;
- macro F1 na zbiorze kontrolnym nie spada poniżej 0,90;
- odpowiedź predykcji ma dokładnie pola z `OUTPUT_KEYS`, a `prediction` jest liczbą całkowitą;
- transformacja jest deterministyczna.

In [ ]:
# Zadanie: uzupełnij zestaw testów.
write_file(
    "tests/test_pipeline.py",
    """
    import pytest
    from sklearn.datasets import load_iris
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split

    from contract import INPUT_SCHEMA, OUTPUT_KEYS
    from features import FEATURE_COLUMNS, TARGET_COLUMN, build_features
    from model import build_model
    from service import predict

    SEED = 42
    QUALITY_THRESHOLD = 0.90


    @pytest.fixture(scope="module")
    def data():
        return load_iris(as_frame=True).frame


    def test_features_do_not_contain_target(data):
        assert TARGET_COLUMN not in build_features(data).columns
    """,
)
run_pytest()

In [ ]:
%%skip True
write_file(
    "tests/test_pipeline.py",
    """
    import numpy as np
    import pandas as pd
    import pytest
    from sklearn.datasets import load_iris
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split

    from contract import INPUT_SCHEMA, OUTPUT_KEYS
    from features import FEATURE_COLUMNS, TARGET_COLUMN, build_features
    from model import build_model
    from service import predict

    SEED = 42
    QUALITY_THRESHOLD = 0.90


    @pytest.fixture(scope="module")
    def data():
        return load_iris(as_frame=True).frame


    @pytest.fixture(scope="module")
    def split(data):
        return train_test_split(
            data[FEATURE_COLUMNS],
            data[TARGET_COLUMN],
            test_size=0.3,
            random_state=SEED,
            stratify=data[TARGET_COLUMN],
        )


    @pytest.fixture(scope="module")
    def trained(split):
        x_train, _, y_train, _ = split
        return build_model().fit(x_train, y_train)


    # --- warstwa: kod ---

    def test_features_do_not_contain_target(data):
        assert TARGET_COLUMN not in build_features(data).columns


    def test_features_are_numeric(data):
        dtypes = build_features(data).dtypes
        assert all(np.issubdtype(dtype, np.floating) for dtype in dtypes), dict(dtypes)


    def test_transform_is_deterministic(data):
        pd.testing.assert_frame_equal(build_features(data), build_features(data))


    # --- warstwa: dane ---

    def test_schema_rejects_wrong_type(data):
        broken = data[FEATURE_COLUMNS].copy()
        broken["petal width (cm)"] = broken["petal width (cm)"].astype(str)
        with pytest.raises(Exception):
            INPUT_SCHEMA.validate(broken)


    def test_schema_rejects_extra_column(data):
        broken = data[FEATURE_COLUMNS].copy()
        broken["id_rekordu_z_systemu"] = range(len(broken))
        with pytest.raises(Exception):
            INPUT_SCHEMA.validate(broken)


    # --- warstwa: model ---

    def test_model_quality_above_threshold(split, trained):
        _, x_test, _, y_test = split
        score = f1_score(y_test, trained.predict(x_test), average="macro")
        assert score >= QUALITY_THRESHOLD, f"macro F1 = {score:.3f} < {QUALITY_THRESHOLD}"


    # --- warstwa: kontrakt ---

    def test_prediction_contract(split, trained):
        _, x_test, _, _ = split
        output = predict(trained, x_test.head(3).to_dict(orient="records"))
        assert len(output) == 3
        for item in output:
            assert set(item) == set(OUTPUT_KEYS), f"pola odpowiedzi: {sorted(item)}"
            assert isinstance(item["prediction"], int), type(item["prediction"])
    """,
)
run_pytest()

## 4. Naprawa defektów

Zestaw jest czerwony i dokładnie wskazuje, co jest nie tak. Teraz naprawiamy kod — po jednym defekcie.

**Zadanie:** popraw wszystkie cztery moduły tak, aby zestaw był zielony. Nie zmieniaj testów.

In [ ]:
# Zadanie: napraw cztery defekty.


In [ ]:
%%skip True
write_file(
    "src/features.py",
    """
    # Transformacja cech.
    from __future__ import annotations

    import pandas as pd

    FEATURE_COLUMNS = [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)",
    ]
    TARGET_COLUMN = "target"


    def build_features(frame: pd.DataFrame) -> pd.DataFrame:
        missing = sorted(set(FEATURE_COLUMNS) - set(frame.columns))
        if missing:
            raise ValueError(f"Brak wymaganych cech: {missing}")
        return frame[FEATURE_COLUMNS].astype(float)
    """,
)

write_file(
    "src/contract.py",
    """
    # Kontrakt wejścia i wyjścia.
    import pandera.pandas as pa

    OUTPUT_KEYS = ("prediction", "model_version")

    INPUT_SCHEMA = pa.DataFrameSchema(
        columns={
            "sepal length (cm)": pa.Column(float, pa.Check.in_range(3.0, 9.0), nullable=False),
            "sepal width (cm)": pa.Column(float, pa.Check.in_range(1.0, 5.0), nullable=False),
            "petal length (cm)": pa.Column(float, pa.Check.in_range(0.5, 8.0), nullable=False),
            "petal width (cm)": pa.Column(float, pa.Check.in_range(0.05, 3.0), nullable=False),
        },
        strict=True,
        coerce=False,
    )
    """,
)

write_file(
    "src/model.py",
    """
    # Definicja modelu.
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import StandardScaler

    SEED = 42


    def build_model() -> Pipeline:
        return Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(C=1.0, max_iter=500, random_state=SEED)),
            ]
        )
    """,
)

write_file(
    "src/service.py",
    """
    # Warstwa predykcji udostępniana na zewnątrz.
    from __future__ import annotations

    import pandas as pd

    from features import build_features

    MODEL_VERSION = "0.1.0"


    def predict(model, records: list[dict]) -> list[dict]:
        frame = build_features(pd.DataFrame.from_records(records))
        return [
            {"prediction": int(value), "model_version": MODEL_VERSION}
            for value in model.predict(frame)
        ]
    """,
)
run_pytest()

## 5. Punkt kontrolny 1 — test kruchy

Piszemy ten sam test jakości bez ustalonego ziarna i z progiem dopasowanym do najlepszego zaobserwowanego wyniku.

**Do opisania:** ile razy test kruchy przeszedł i dlaczego taki test jest gorszy niż brak testu?

In [ ]:
write_file(
    "tests/test_flaky_demo.py",
    """
    from sklearn.datasets import load_iris
    from sklearn.metrics import f1_score
    from sklearn.model_selection import train_test_split

    from features import FEATURE_COLUMNS, TARGET_COLUMN
    from model import build_model


    def test_quality_flaky():
        frame = load_iris(as_frame=True).frame
        # Brak random_state oraz próg dopasowany do najlepszego zaobserwowanego wyniku.
        x_train, x_test, y_train, y_test = train_test_split(
            frame[FEATURE_COLUMNS], frame[TARGET_COLUMN], test_size=0.3
        )
        model = build_model().fit(x_train, y_train)
        assert f1_score(y_test, model.predict(x_test), average="macro") >= 0.98
    """,
)

results = [run_pytest("tests/test_flaky_demo.py") for _ in range(5)]
print(f"\nPrzeszedł {results.count(0)} z 5 uruchomień tego samego, niezmienionego kodu.")

(PROJECT_DIR / "tests" / "test_flaky_demo.py").unlink()
print("Test kruchy usunięty z zestawu.")

## 6. Punkt kontrolny 2 — testy zachowania modelu

Metryka nie sprawdza, czy model zachowuje się sensownie. Dopisujemy dwa testy:

- **niezmienniczość** — dodanie kolumny technicznej nie zmienia predykcji;
- **kierunkowość** — zwiększanie długości i szerokości płatka nie może obniżać przewidywanego numeru klasy (w zbiorze Iris klasy rosną wraz z rozmiarem płatka).

**Zadanie:** napisz plik `tests/test_behaviour.py` z tymi dwoma testami.

In [ ]:
# Zadanie: dopisz testy zachowania modelu.


In [ ]:
%%skip True
write_file(
    "tests/test_behaviour.py",
    """
    import pytest
    from sklearn.datasets import load_iris

    from features import FEATURE_COLUMNS, TARGET_COLUMN
    from model import build_model
    from service import predict

    SEED = 42


    @pytest.fixture(scope="module")
    def trained():
        frame = load_iris(as_frame=True).frame
        return build_model().fit(frame[FEATURE_COLUMNS], frame[TARGET_COLUMN])


    def test_invariance_to_technical_column(trained):
        frame = load_iris(as_frame=True).frame.head(20)
        plain = frame[FEATURE_COLUMNS].to_dict(orient="records")
        with_extra = [
            {**record, "id_rekordu_z_systemu": index} for index, record in enumerate(plain)
        ]
        assert predict(trained, plain) == predict(trained, with_extra)


    def test_direction_of_petal_size(trained):
        records = [
            {
                "sepal length (cm)": 5.8,
                "sepal width (cm)": 3.0,
                "petal length (cm)": length,
                "petal width (cm)": width,
            }
            for length, width in [(1.4, 0.2), (4.0, 1.2), (5.5, 2.0), (6.5, 2.4)]
        ]
        classes = [item["prediction"] for item in predict(trained, records)]
        assert classes == sorted(classes), classes
    """,
)
run_pytest()

## 7. Testy własności

Wszystkie dotychczasowe testy mają wspólną słabość: sprawdzają **przypadki, które przyszły nam do głowy**. Defekt czeka zwykle tam, gdzie nikt nie pomyślał — przy wartości granicznej, przy jednym rekordzie, przy setce identycznych rekordów.

Test własności odwraca perspektywę. Zamiast podawać dane, opisujemy **jakie dane są dopuszczalne**, a biblioteka generuje z tego opisu setki przypadków i sama szuka kontrprzykładu. Gdy go znajdzie, upraszcza go do najmniejszej postaci, która wciąż psuje test.

Dobre własności dla usługi predykcyjnej:

| Własność | Treść |
|---|---|
| **zachowanie długości** | liczba odpowiedzi równa się liczbie żądań |
| **zgodność kontraktu** | każda odpowiedź ma dokładnie wymagane pola i typy |
| **determinizm** | to samo wejście dwa razy daje ten sam wynik |
| **niezależność od kolejności** | permutacja rekordów permutuje odpowiedzi, nic więcej |

Żadna z nich nie mówi, **co** model ma przewidzieć. Wszystkie mówią, jak system ma się zachowywać — i właśnie dlatego nie trzeba ich poprawiać po każdym ponownym treningu.

**Zadanie:** dopisz `tests/test_properties.py` z testami własności przy użyciu biblioteki `hypothesis`.

In [ ]:
# Zadanie: dopisz testy własności.


In [ ]:
%%skip True
write_file(
    "tests/test_properties.py",
    """
    # Testy wlasnosci: opisujemy dopuszczalne dane, a nie konkretne przypadki.
    import pytest
    from hypothesis import HealthCheck, given, settings
    from hypothesis import strategies as st
    from sklearn.datasets import load_iris

    from contract import OUTPUT_KEYS
    from features import FEATURE_COLUMNS, TARGET_COLUMN
    from model import build_model
    from service import predict

    # Zakresy odpowiadaja kontraktowi danych, a nie temu, co akurat jest w zbiorze.
    RANGES = {
        "sepal length (cm)": (3.0, 9.0),
        "sepal width (cm)": (1.0, 5.0),
        "petal length (cm)": (0.5, 8.0),
        "petal width (cm)": (0.05, 3.0),
    }

    records = st.lists(
        st.fixed_dictionaries(
            {
                name: st.floats(
                    min_value=low, max_value=high, allow_nan=False, allow_infinity=False
                )
                for name, (low, high) in RANGES.items()
            }
        ),
        min_size=1,
        max_size=25,
    )


    @pytest.fixture(scope="module")
    def trained():
        frame = load_iris(as_frame=True).frame
        return build_model().fit(frame[FEATURE_COLUMNS], frame[TARGET_COLUMN])


    # Model trenujemy raz na modul, wiec wylaczamy ostrzezenie o wolnym fixture.
    SETTINGS = settings(
        max_examples=50, deadline=None, suppress_health_check=[HealthCheck.function_scoped_fixture]
    )


    @SETTINGS
    @given(batch=records)
    def test_length_is_preserved(trained, batch):
        assert len(predict(trained, batch)) == len(batch)


    @SETTINGS
    @given(batch=records)
    def test_output_contract_holds(trained, batch):
        for answer in predict(trained, batch):
            assert set(answer) == set(OUTPUT_KEYS)
            assert isinstance(answer["prediction"], int)


    @SETTINGS
    @given(batch=records)
    def test_prediction_is_deterministic(trained, batch):
        assert predict(trained, batch) == predict(trained, batch)


    @SETTINGS
    @given(batch=records)
    def test_order_does_not_change_results(trained, batch):
        forward = predict(trained, batch)
        backward = predict(trained, list(reversed(batch)))
        assert forward == list(reversed(backward))
    """,
)

run_pytest("tests/test_properties.py")

## 8. Kontrprzykład znaleziony przez maszynę

Do usługi wraca „drobna poprawka": ktoś dopisał filtr odrzucający rekordy o bardzo małej szerokości płatka, uznając je za błąd pomiaru. Najmniejsza wartość w zbiorze uczącym to 0,1, więc **wszystkie dotychczasowe testy nadal przechodzą** — używają przecież prawdziwych danych.

Kontrakt danych dopuszcza jednak wartości od 0,05. `hypothesis` generuje wartości graniczne priorytetowo, więc trafia w ten przedział natychmiast.

**Do opisania:** do jakiej postaci biblioteka uprościła kontrprzykład i dlaczego akurat do takiej?

In [ ]:
service_path = PROJECT_DIR / "src" / "service.py"
original_service = service_path.read_text(encoding="utf-8")

# Zmiana wyglada niewinnie: odrzucamy "bledne pomiary" o bardzo malym platku.
ANCHOR = "    frame = build_features(pd.DataFrame.from_records(records))"
FILTER = '    frame = frame[frame["petal width (cm)"] >= 0.1]'

assert original_service.count(ANCHOR) == 1, "Nie znaleziono miejsca wstrzyknięcia defektu."
service_path.write_text(
    original_service.replace(ANCHOR, ANCHOR + "\n" + FILTER),
    encoding="utf-8",
)

print("Testy oparte na prawdziwych danych — nadal zielone:")
run_pytest("tests/test_behaviour.py", "-q")

print("\nTen sam kod pod testem własności:")
run_pytest("tests/test_properties.py::test_length_is_preserved", "-q")

service_path.write_text(original_service, encoding="utf-8")
print("\nPrzywrócono poprzednią wersję usługi.")
print(
    "Najmniejsza szerokość płatka w zbiorze uczącym to 0.1, więc testy na prawdziwych\n"
    "danych nigdy nie dotknęły tego progu. Kontrakt dopuszcza 0.05 — i dokładnie tam\n"
    "hypothesis szukała najpierw, bo wartości graniczne generuje priorytetowo."
)

## 9. Punkt kontrolny 3 — czas wykonania zestawu

Zestaw, który wykonuje się długo, przestaje być uruchamiany. Sprawdzamy, które testy są najwolniejsze, i pokazujemy mechanizm oznaczania testów wolnych.

**Do opisania:** który test jest najwolniejszy i co zrobisz, gdy zestaw przekroczy dwie minuty?

In [ ]:
run_pytest("--durations=5")

print("\nSzybki podzbiór (bez testów oznaczonych jako slow):")
run_pytest("-m", "not slow")

## 10. Podsumowanie

Projekt ma teraz siatkę bezpieczeństwa obejmującą cztery warstwy: kod, dane, model i kontrakt. Zestaw uruchamia się jedną komendą i kończy w kilka sekund, więc będzie faktycznie używany.

Czego nadal brakuje:

- testy uruchamia człowiek, a nie automat na czystej maszynie (zajęcia 9);
- model nie ma wersji ani sygnatury, więc kontrakt jest pilnowany tylko w testach (zajęcia 7);
- nie istnieje usługa, przeciwko której można by uruchomić test kontraktu end-to-end (zajęcia 8).

### Zadanie do własnego projektu

1. Dwa testy jednostkowe transformacji cech, w tym jeden przypadek brzegowy.
2. Test kontraktu danych odrzucającego błędny typ i kolumnę nadmiarową.
3. Test progu jakości na ustalonym, wersjonowanym zbiorze kontrolnym.
4. Jeden test zachowania modelu — niezmienniczość albo kierunkowość.
5. Test formatu wyjścia predykcji.
6. Uruchomienie zestawu jedną komendą, poniżej dwóch minut.
7. Opis w `README.md`, przed czym chroni każda grupa testów.